# Análisis: libros más vendidos de Amazon (2009-2019)

**Preguntas que responde este notebook:**
1. ¿Qué libros y autores aparecen más veces en el top 50?
2. ¿Cómo se comparan ficción y no ficción?
3. ¿Cómo evolucionaron género, precio y valoración a lo largo de los años?
4. ¿Hay relación entre precio, valoración y número de reseñas?

**Nota metodológica:** el dataset no contiene unidades vendidas. "Más vendido" se interpreta como "más presente en el top 50 anual de Amazon".

In [1]:
# Importar librerías
import pandas as pd

# Mostrar todas las columnas al imprimir tablas
pd.set_option("display.max_columns", None)

# Cargar el dataset limpio desde data/processed
print("=== CARGA DEL DATASET LIMPIO ===")
df = pd.read_csv("../data/processed/amazon_bestsellers_clean.csv")

print("Dimensiones (filas, columnas):", df.shape)
print("Columnas:", list(df.columns))

# Subconjunto sin libros gratuitos, para análisis de precios
# (el precio 0 distorsionaría los promedios)
df_pago = df[~df["is_free"]]
print("Filas totales:", len(df), "| Filas de pago:", len(df_pago))

=== CARGA DEL DATASET LIMPIO ===
Dimensiones (filas, columnas): (550, 8)
Columnas: ['name', 'author', 'user_rating', 'reviews', 'price', 'year', 'genre', 'is_free']
Filas totales: 550 | Filas de pago: 538


In [2]:
# Panorama general del dataset
print("=== PANORAMA GENERAL ===")

# Contar libros únicos (no filas), porque un título puede repetirse entre años
print("Apariciones totales (filas):", len(df))
print("Libros únicos:", df["name"].nunique())
print("Autores únicos:", df["author"].nunique())

# Distribución por género (en apariciones)
print("\n--- Apariciones por género ---")
conteo_genero = df["genre"].value_counts()
porcentaje_genero = df["genre"].value_counts(normalize=True).mul(100).round(1)
display(pd.DataFrame({"apariciones": conteo_genero, "porcentaje": porcentaje_genero}))

# Distribución de libros únicos por género
print("\n--- Libros únicos por género ---")
display(df.groupby("genre")["name"].nunique())

=== PANORAMA GENERAL ===
Apariciones totales (filas): 550
Libros únicos: 351
Autores únicos: 248

--- Apariciones por género ---


,apariciones,porcentaje
genre,,
Non Fiction,310,56.4
Fiction,240,43.6



--- Libros únicos por género ---


genre
Fiction        160
Non Fiction    191
Name: name, dtype: int64

In [3]:
# PREGUNTA 1A: ¿Qué libros aparecen más veces en el top 50?
print("=== LIBROS CON MÁS APARICIONES EN EL TOP 50 ===")

# Agrupar por título y autor, contando en cuántas filas (años) aparece cada libro
top_libros = (
    df.groupby(["name", "author", "genre"])
    .size()
    .reset_index(name="apariciones")
    .sort_values("apariciones", ascending=False)
    .head(10)
)
display(top_libros)

# Cuántos libros aparecieron una sola vez
apariciones_por_libro = df["name"].value_counts()
una_vez = (apariciones_por_libro == 1).sum()
print(f"\nLibros que aparecen una sola vez: {una_vez} de {len(apariciones_por_libro)}")
# print(f"Porcentaje: {una_vez / len(apariciones_por_libro) * 100:.1f}%")

=== LIBROS CON MÁS APARICIONES EN EL TOP 50 ===


,name,author,genre,apariciones
191,Publication Manual of the American Psychologic...,American Psychological Association,Non Fiction,10
209,StrengthsFinder 2.0,Gallup,Non Fiction,9
178,"Oh, the Places You'll Go!",Dr. Seuss,Fiction,8
310,The Very Hungry Caterpillar,Eric Carle,Fiction,7
219,The 7 Habits of Highly Effective People: Power...,Stephen R. Covey,Non Fiction,7
243,The Four Agreements: A Practical Guide to Pers...,Don Miguel Ruiz,Non Fiction,6
140,Jesus Calling: Enjoying Peace in His Presence ...,Sarah Young,Non Fiction,6
281,The Official SAT Study Guide,The College Board,Non Fiction,5
216,The 5 Love Languages: The Secret to Love That ...,Gary Chapman,Non Fiction,5
152,Laugh-Out-Loud Jokes for Kids,Rob Elliott,Non Fiction,5



Libros que aparecen una sola vez: 255 de 351


In [4]:
# PREGUNTA 1B: ¿Qué autores dominan el ranking?
print("=== AUTORES CON MÁS PRESENCIA EN EL TOP 50 ===")

# Para cada autor: total de apariciones y cuántos títulos distintos tiene
top_autores = (
    df.groupby("author")
    .agg(
        apariciones=("name", "size"),
        libros_unicos=("name", "nunique"),
        rating_promedio=("user_rating", "mean"),
    )
    .round(2)
    .sort_values("apariciones", ascending=False)
    .head(10)
)
display(top_autores)

# Autores con mayor cantidad de títulos distintos en el top
print("\n--- Autores con más títulos distintos ---")
display(top_autores.sort_values("libros_unicos", ascending=False).head(5))

=== AUTORES CON MÁS PRESENCIA EN EL TOP 50 ===


,apariciones,libros_unicos,rating_promedio
author,,,
Jeff Kinney,12,12,4.80
Gary Chapman,11,3,4.74
Rick Riordan,11,10,4.77
Suzanne Collins,11,5,4.66
American Psychological Association,10,1,4.50
Dr. Seuss,9,2,4.88
Gallup,9,1,4.00
Rob Elliott,8,2,4.56
Dav Pilkey,7,6,4.90



--- Autores con más títulos distintos ---


,apariciones,libros_unicos,rating_promedio
author,,,
Jeff Kinney,12,12,4.80
Rick Riordan,11,10,4.77
Dav Pilkey,7,6,4.90
Bill O'Reilly,7,6,4.64
Suzanne Collins,11,5,4.66


In [5]:
# PREGUNTA 2: ¿Cómo se comparan ficción y no ficción?
print("=== COMPARACIÓN FICCIÓN VS NO FICCIÓN ===")

# Métricas generales por género
comparativa = df.groupby("genre").agg(
    apariciones=("name", "size"),
    libros_unicos=("name", "nunique"),
    rating_promedio=("user_rating", "mean"),
    reviews_mediana=("reviews", "median"),  # mediana: las reseñas tienen valores extremos
).round(2)
display(comparativa)

# Precio por género, excluyendo libros gratuitos
print("\n--- Precio por género (sin libros gratuitos) ---")
display(df_pago.groupby("genre")["price"].agg(["mean", "median", "max"]).round(2))

=== COMPARACIÓN FICCIÓN VS NO FICCIÓN ===


,apariciones,libros_unicos,rating_promedio,reviews_mediana
genre,,,,
Fiction,240,160,4.65,10922.0
Non Fiction,310,191,4.60,6346.0



--- Precio por género (sin libros gratuitos) ---


,mean,median,max
genre,,,
Fiction,11.37,9.0,82
Non Fiction,14.89,12.0,105


In [6]:
# PREGUNTA 3: ¿Cómo evolucionaron los datos año a año?
print("=== EVOLUCIÓN POR AÑO ===")

# Cantidad de apariciones por año y género
print("--- Apariciones por año y género ---")
por_anio_genero = pd.crosstab(df["year"], df["genre"])
display(por_anio_genero)

# Porcentaje de no ficción por año
print("\n--- % de No Ficción por año ---")
display(por_anio_genero.div(por_anio_genero.sum(axis=1), axis=0).mul(100).round(1))

# Precio promedio por año (sin libros gratuitos)
print("\n--- Precio promedio por año (sin gratuitos) ---")
display(df_pago.groupby("year")["price"].mean().round(2))

# Valoración promedio por año
print("\n--- Valoración promedio por año ---")
display(df.groupby("year")["user_rating"].mean().round(3))

=== EVOLUCIÓN POR AÑO ===
--- Apariciones por año y género ---


genre,Fiction,Non Fiction
year,,
2009,24,26
2010,20,30
2011,21,29
2012,21,29
2013,24,26
2014,29,21
2015,17,33
2016,19,31
2017,24,26



--- % de No Ficción por año ---


genre,Fiction,Non Fiction
year,,
2009,48.0,52.0
2010,40.0,60.0
2011,42.0,58.0
2012,42.0,58.0
2013,48.0,52.0
2014,58.0,42.0
2015,34.0,66.0
2016,38.0,62.0
2017,48.0,52.0



--- Precio promedio por año (sin gratuitos) ---


year
2009    15.40
2010    13.76
2011    15.41
2012    15.30
2013    15.21
2014    15.91
2015    10.63
2016    13.73
2017    11.61
2018    10.52
2019    10.08
Name: price, dtype: float64


--- Valoración promedio por año ---


year
2009    4.584
2010    4.558
2011    4.558
2012    4.532
2013    4.554
2014    4.622
2015    4.648
2016    4.678
2017    4.660
2018    4.668
2019    4.740
Name: user_rating, dtype: float64

In [7]:
# PREGUNTA 4: ¿Hay relación entre precio, valoración y reseñas?
print("=== CORRELACIONES ===")

# Spearman en vez de Pearson: las reseñas y precios tienen valores extremos
columnas_num = ["user_rating", "reviews", "price", "year"]
correlaciones = df_pago[columnas_num].corr(method="spearman").round(2)
display(correlaciones)

# Libros con más reseñas (una fila por título, la de mayor cantidad de reseñas)
print("\n--- Los 10 libros con más reseñas ---")
mas_resenas = (
    df.sort_values("reviews", ascending=False)
    .drop_duplicates(subset="name")
    .head(10)[["name", "author", "year", "reviews", "user_rating"]]
)
display(mas_resenas)

=== CORRELACIONES ===


,user_rating,reviews,price,year
user_rating,1.00,0.20,-0.21,0.29
reviews,0.20,1.00,-0.17,0.40
price,-0.21,-0.17,1.00,-0.22
year,0.29,0.40,-0.22,1.00



--- Los 10 libros con más reseñas ---


,name,author,year,reviews,user_rating
534,Where the Crawdads Sing,Delia Owens,2019,87841,4.8
382,The Girl on the Train,Paula Hawkins,2015,79446,4.1
33,Becoming,Michelle Obama,2019,61133,4.8
136,Gone Girl,Gillian Flynn,2013,57271,4.0
368,The Fault in Our Stars,John Green,2014,50482,4.7
437,The Nightingale: A Novel,Kristin Hannah,2015,49288,4.8
106,Fifty Shades of Grey: Book One of the Fifty Sh...,E L James,2012,47265,3.8
433,The Martian,Andy Weir,2015,39459,4.7
20,All the Light We Cannot See,Anthony Doerr,2014,36348,4.6
338,The Alchemist,Paulo Coelho,2014,35799,4.7
